In [2]:
# Precison lost example


import torch

x = torch.tensor([1.001])

for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    print(dtype, x.to(dtype).item())


torch.float32 1.0010000467300415
torch.float16 1.0009765625
torch.bfloat16 1.0


# Numerical Precision in Deep Learning: From FP32 to 4-Bit Quantization

*A first-principles guide to floating-point formats, quantization, and mixed-precision training.*

## 0. The Big Picture

Suppose a neural network has **1 billion parameters**.

How much memory do we need just to store its weights?

| Format | Bits per parameter | Theoretical weight memory |
|---|---:|---:|
| FP32 | 32 | 4 GB |
| FP16 | 16 | 2 GB |
| BF16 | 16 | 2 GB |
| 4-bit | 4 | 0.5 GB |

These numbers exclude metadata, activations, optimizer states, and other memory overhead.

This simple observation leads to a much deeper question:

**Do neural networks really need 32 bits of precision for every parameter?**

To answer that, we need to distinguish three concepts:

1. **Numerical Representation:** How does a computer represent a number?
2. **Quantization:** How do we approximate numbers using fewer bits?
3. **Mixed-Precision Computing:** How do we use different numerical precisions in different parts of a computation?

These concepts are related, but they are not interchangeable.

---

## 1. What Does FP32 Actually Mean? Is 4-Bit Always FP4?

**Question:** If FP32 means 32 bits, does FP4 simply mean 4 bits?

### First Principles

FP stands for **Floating Point**.

The number indicates the total number of bits used to represent one value.

For example:

- FP32: 32-bit floating-point format.
- FP16: 16-bit floating-point format.
- FP8: 8-bit floating-point format.
- FP4: 4-bit floating-point format.

However, knowing the number of bits does not fully define the encoding.

A 4-bit value has:

`2^4 = 16`

possible binary patterns.

Those patterns could represent integers, floating-point values, or entries in a custom lookup table.

### Example

The same binary pattern, `0011`, could mean:

| Format | Interpretation |
|---|---|
| Unsigned INT4 | 3 |
| Signed INT4 (two's complement) | 3 |
| FP4 E2M1 | 1.5 |
| Custom codebook | Whatever value the codebook assigns |

**Key insight:** Bits store patterns. Encoding rules give those patterns numerical meaning.

### Naming Cheat Sheet

| Name | Meaning | Typical structure |
|---|---|---|
| FP32 | 32-bit floating point | E8M23 |
| FP16 | 16-bit floating point | E5M10 |
| BF16 | 16-bit Brain Floating Point | E8M7 |
| FP8 | 8-bit floating point | E4M3, E5M2, etc. |
| FP4 | 4-bit floating point | E2M1, for example |
| INT4 | 4-bit integer | Signed or unsigned |
| NF4 | 4-bit NormalFloat | Nonuniform codebook |

Here:

- **S:** Sign
- **E:** Exponent
- **M:** Mantissa (fraction bits)

For these floating-point formats:

`Total bits = 1 sign bit + exponent bits + mantissa bits`

**Mental model:** `FP16` tells us the storage width. `E5M10` tells us how the bits are allocated. Additional rules define how the values are decoded.

---

## 2. Why Is Floating Point Designed This Way?

**Question:** Why do we need strange rules like exponent bias and `1 + M/2`?

### Problem: Representing Very Different Magnitudes

Imagine we want to represent:

- 1,234
- 0.001234
- 12,340,000

A fixed decimal-point position is inconvenient.

Scientific notation solves this:

`1.234 × 10^3`

`1.234 × 10^-3`

`1.234 × 10^7`

We keep the significant digits and change the exponent.

Binary floating point follows the same idea, but uses base 2 instead of base 10.

For normal binary floating-point values:

`Value = (-1)^S × Significand × 2^(Exponent)`

### Why the Hidden Leading 1?

Consider binary scientific notation:

`101.1₂ = 1.011₂ × 2²`

For every nonzero normalized binary number, the significand begins with `1`.

Why store that bit if it is always predictable?

Instead, we store only the bits after the binary point.

For a one-bit mantissa:

- `M = 0` means `1.0₂ = 1`
- `M = 1` means `1.1₂ = 1.5`

Therefore:

`Significand = 1 + M/2`

This is called the **implicit or hidden leading 1**.

It effectively saves one bit.

### Why Exponent Bias?

Scientific notation needs negative exponents:

`0.125 = 1 × 2^-3`

But we can store the exponent bits as an unsigned integer and subtract a fixed bias:

`Actual exponent = Stored exponent - Bias`

For the usual IEEE-style exponent convention:

`Bias = 2^(E_bits - 1) - 1`

For FP32, with 8 exponent bits:

`Bias = 127`

For FP4 E2M1, with 2 exponent bits:

`Bias = 1`

This allows unsigned exponent codes to represent both negative and positive powers of two.

### Why Subnormal Numbers?

If every encoded number used the hidden leading 1, we could not represent zero.

So floating-point formats reserve special exponent patterns.

One use is to represent zero and **subnormal numbers**, which are extremely small values without the implicit leading 1.

**Mental model:**

- Exponent: controls magnitude.
- Mantissa: controls precision.
- Exponent bias: encodes positive and negative exponents.
- Hidden leading 1: avoids storing a predictable bit.
- Subnormal numbers: provide values close to zero.

These rules are engineering solutions to the problem of limited bits.

---

## 3. A Complete FP4 E2M1 Example

**Question:** Given exactly four bits, how do we calculate the number? What are the maximum and minimum values?

Consider FP4 E2M1, the finite-only FP4 format used in NVIDIA's FP4 implementations.

Its bit layout is:

`S | EE | M`

Let's decode:

`0101`

### Step 1: Sign

`S = 0`

Therefore, the number is positive.

### Step 2: Exponent

`EE = 10₂ = 2`

Bias = 1.

`Actual exponent = 2 - 1 = 1`

### Step 3: Mantissa

`M = 1`

The significand is:

`1 + 1/2 = 1.5`

### Step 4: Calculate

`Value = (+1) × 1.5 × 2^1 = 3.0`

Therefore:

**FP4 E2M1 `0101` represents 3.0.**

### The Complete E2M1 Value Table

| Binary | Value | Binary | Value |
|---|---:|---|---:|
| 0000 | +0 | 1000 | -0 |
| 0001 | +0.5 | 1001 | -0.5 |
| 0010 | +1 | 1010 | -1 |
| 0011 | +1.5 | 1011 | -1.5 |
| 0100 | +2 | 1100 | -2 |
| 0101 | +3 | 1101 | -3 |
| 0110 | +4 | 1110 | -4 |
| 0111 | +6 | 1111 | -6 |

For the `E = 00` patterns, E2M1 uses the special subnormal/zero rule instead of the normal hidden-leading-1 rule.

Therefore:

- Maximum: **+6**
- Minimum: **-6**
- Smallest positive nonzero value: **0.5**

We trade 5,7 with 0.5 and 1.5

The encoding has 16 patterns but 15 distinct numerical values, since +0 and -0 are numerically equal.

These limits describe the raw format, before applying an external scale.

---

## 4. INT4: Isn't It Just Ordinary Binary?

**Question:** If four bits are already occupied by the integer, where does the scale go?

### First Principles

INT4 represents integers using four bits.

For conventional signed two's-complement INT4, the range is:

`-8, -7, ..., 0, ..., 6, 7`

But neural network weights are often fractional:

`W = [0.3, -0.2, 0.7, 0.0]`

We cannot represent these numbers directly with integers.

A simple solution is to introduce a **scale**.

Choose:

`scale = 0.1`

Then quantize:

`Q = [3, -2, 7, 0]`

Reconstruct approximate original values using:

`W ≈ Q × scale`

This is symmetric integer quantization.

### Where Is the Scale Stored?

**Separately from the 4-bit integer codes.**

For example:

- Four INT4 weights: 16 bits.
- One shared FP16 scale: 16 bits.
- Total: 32 bits.

The scale is additional metadata, not part of the four-bit encoding.

To reduce this overhead, many weights can share one scale.

Suppose 64 weights share a 16-bit scale:

`64 × 4 + 16 = 272 bits`

Average:

`272 / 64 = 4.25 bits per weight`

That is close to the theoretical 4-bit limit.

Real quantization systems may also use zero-points, group-level metadata, or additional scale levels.

### Mental Model

Think of scale as choosing the unit of a ruler.

Without scale:

`3` means three units.

With scale = 0.1:

`3` means 0.3 units.

We do not need to store the ruler's unit beside every measurement if an entire group shares the same unit.

---

## 5. What Is NF4? Is There an NF32?

**Question:** Is NormalFloat a general floating-point family like FP16 and FP32?

Not really.

NF4 stands for **NormalFloat 4-bit**.

It was introduced in the QLoRA context for quantizing neural network weights whose distributions are approximately normal.

### Why Not Just Use Uniform Spacing?

Suppose 90% of weights lie between -0.1 and +0.1.

If we only have 16 representable levels, should we distribute all 16 evenly across the full range?

Not necessarily.

We might achieve smaller average quantization error by placing more levels where weights are concentrated.

That is the core intuition behind nonuniform quantization.

NF4 uses a specially designed 16-value codebook, combined with scaling.

It is not simply the standard floating-point sign/exponent/mantissa layout.

### Is There NF32?

There is no widely adopted standard called NF32 analogous to FP32.

NF4 is the name of a specific 4-bit format. It does not imply a standardized NF8/NF16/NF32 family.

**Mental model:** FP formats are based on floating-point exponent and significand rules. NF4 is a specialized codebook-based numerical representation.

---

## 6. How Much Precision Do FP32, FP16, and BF16 Actually Provide?

**Question:** What is the real numerical difference? Can we measure it?

Let's use the number:

`1.001`

Different formats store approximately:

| Format | Stored value |
|---|---:|
| FP32 | 1.001000047 |
| FP16 | 1.000976563 |
| BF16 | 1.000000000 |

BF16 rounds 1.001 to 1.0 because the difference is too small for its available significand precision.

### Spacing Near 1.0

Another useful measure is the distance from 1.0 to the next larger representable number.

| Format | Next number after 1.0 | Gap |
|---|---:|---:|
| FP32 | 1.0000001192 | 0.0000001192 |
| FP16 | 1.0009765625 | 0.0009765625 |
| BF16 | 1.0078125 | 0.0078125 |
| FP4 E2M1 | 1.5 | 0.5 |

This distance is related to the concept of **ULP (Unit in the Last Place)**.

The gap is not constant over the entire number line.

Floating-point resolution changes with the exponent.

A subtle detail: BF16's next smaller value below 1.0 is 0.99609375, while its next larger value is 1.0078125.

### Range vs. Precision

There are two independent questions:

1. What is the largest or smallest magnitude a format can represent?
2. How finely can it distinguish nearby numbers?

| Format | Exponent bits | Fraction bits | Approx. maximum finite positive value |
|---|---:|---:|---:|
| FP32 | 8 | 23 | 3.4 × 10^38 |
| FP16 | 5 | 10 | 65,504 |
| BF16 | 8 | 7 | 3.39 × 10^38 |

FP16 has more precision than BF16, but a much narrower range.

BF16 sacrifices significand precision to preserve a range similar to FP32.

**Mental model:**

Exponent is how far your ruler can reach.

Mantissa is how finely your ruler is marked.

---

## 7. Aren't Neural Network Weights Normalized and Close to 1?

**Question:** If normalization keeps values around 1, why would we need such enormous numerical ranges?

The key misunderstanding is:

**Normalizing activations does not mean normalizing all model weights to 1.**

Consider:

`y = Wx + b`

We might apply LayerNorm to the activations `x`.

For example:

`x = [10, 20, 30]`

After normalization, approximately:

`x_normalized = [-1.22, 0, 1.22]`

But this says nothing about the values of `W`.

Weights may be:

`W = [0.002, -0.15, 0.8]`

Also, normalization to zero mean and unit variance does not mean every activation is near +1.

It means something about the distribution, not every individual value.

Many large weight matrices have values concentrated relatively close to zero. Some normalization layers also have learned scale parameters often initialized to 1.

These are different kinds of parameters.

### Does a Small Weight Mean Precision Is Unimportant?

No.

Suppose:

`w = 0.001`

With integer quantization using scale = 0.01:

`0.001 -> 0`

The weight disappears entirely.

But with a suitable floating-point format or a smaller quantization scale, we may preserve it.

**What matters is not just the magnitude of a weight, but the error relative to its value and the effect of that error on the model's output.**

---

## 8. Why Can a 4-Bit Model Still Work?

**Question:** If 4-bit has so few possible values, why doesn't quantization destroy the model?

### First Principles

The objective of a neural network is not to preserve its parameters exactly.

The objective is to produce useful outputs.

Suppose:

`y = w1*x1 + w2*x2`

With:

`x = [1, 1]`

`w = [0.1234, 0.5678]`

Original output:

`y = 0.6912`

Now approximate the weights:

`w_quantized = [0.12, 0.57]`

New output:

`y_quantized = 0.69`

The output error is only:

`|0.6912 - 0.69| = 0.0012`

Even though both weights changed, the output changed very little.

This is a toy rounding example, not a specific INT4 or FP4 encoder.

### Why This Can Work in Large Models

Neural networks often tolerate some parameter perturbations.

However, not all errors are equally harmless.

A small error might be negligible in one layer but strongly amplified in another.

A small change to logits may also affect the next-token choice if two candidates have almost equal scores.

Therefore, quantization quality depends on:

- Quantization format.
- Scale selection.
- Group size.
- Outliers.
- Sensitivity of different layers.
- Model architecture and task.
- Whether the model is calibrated or fine-tuned for quantization.

A 4-bit model may perform close to its higher-precision version, but this is not guaranteed.

### Memory vs. Model Quality vs. Speed

These are different metrics.

- **Memory:** How many bytes are required?
- **Quality:** How much does quantization change model predictions?
- **Speed:** Can the hardware execute the resulting computation efficiently?

4-bit almost always reduces raw weight storage compared with FP16/FP32.

But it does not automatically guarantee faster inference or equal model quality.

---

## 9. Why Store Weights in 4-Bit but Compute in BF16?

**Question:** If Hugging Face bitsandbytes stores weights in 4-bit, why does it convert them to BF16 for computation? Isn't that defeating the purpose?

### First Principles

Storage and computation are different problems.

A low-precision weight might be sufficient for storing an approximate parameter value.

But intermediate mathematical results may require greater numerical range or precision.

Consider raw FP4 E2M1:

`0.5 × 0.5 = 0.25`

E2M1 can represent 0.5, but cannot directly represent 0.25 without additional scaling.

If we force the result back into raw E2M1, information is lost.

One solution is to perform computation in a wider numerical format.

### Example: A 4-Bit Weight Matrix

Suppose we store a matrix with 4-bit quantized weights.

During inference:

1. Read a block of compressed 4-bit weights.
2. Dequantize those weights into values usable by the computation kernel, possibly BF16.
3. Multiply them by activations.
4. Accumulate the products, often using an appropriate wider precision.
5. Keep the stored model weights compressed.

In implementations such as bitsandbytes, BF16 can be selected as the compute dtype for 4-bit quantized layers.

Crucially:

**Converting a quantized weight to BF16 does not restore information lost during quantization.**

It just gives subsequent operations a more suitable format.

Some specialized hardware can also operate directly on low-bit inputs without a separate full BF16 conversion step.

### Three Independent Precision Choices

| Precision | Question |
|---|---|
| Storage precision | How are model weights kept in memory? |
| Compute precision | What format is used for multiplication and other operations? |
| Accumulation precision | How are many intermediate results summed? |

An illustrative configuration is:

`INT4 storage -> BF16 compute -> FP32 accumulation`

This is an example, not a universal rule for all hardware.

**Mental model:** Compress data when storing it; use the numerical representation appropriate for each calculation.

---

## 10. Why Do We Need BF16 If Weights Are Usually Small?

**Question:** FP16 is more precise than BF16. Why sacrifice precision to obtain a numerical range that weights might never need?

This question reveals an important distinction:

**Training involves much more than storing weights.**

We also have:

- Activations.
- Gradients.
- Losses.
- Intermediate products and sums.
- Optimizer states.

Even if weights are small, intermediate values may be much larger or much smaller.

### Example A: Overflow

Suppose:

`w = 0.5`

`x = 600`

Then:

`y = wx = 300`

Now consider a squared loss with target zero:

`L = y^2 = 90,000`

FP16's maximum finite value is 65,504.

If the squared result is stored in FP16, it overflows to infinity.

BF16 can represent values of this magnitude.

It will round 90,000 to approximately 90,112, but it does not overflow.

This is an illustrative example, not a claim that Transformer activations usually equal 600.

### Example B: Underflow

Suppose a gradient is:

`g = 1e-8`

The smallest positive nonzero FP16 value is approximately:

`5.96e-8`

So FP16 rounds `1e-8` to zero.

BF16 has much greater exponent range and can represent a nonzero value close to `1e-8`.

This is one reason BF16 is attractive for training.

FP16 training can mitigate small-gradient underflow using techniques such as **loss scaling**.

### The Important Conclusion

BF16 is not primarily about having enormous weight values.

It is about accommodating a wide range of **intermediate numerical values during training** while still using 16 bits.

---

## 11. If BF16 Is So Imprecise, How Can We Train an LLM?

**Question:** BF16 rounds 1.001 to 1.0. How can gradient descent make tiny parameter updates?

This is one of the most important questions in numerical deep learning.

### Problem: Small Updates Disappear

Suppose:

`w = 1.0`

Gradient descent wants to update the weight by:

`delta = -0.0001`

Mathematically:

`w_new = 0.9999`

But BF16 rounds 0.9999 back to 1.0.

If we repeatedly update a BF16 weight this way, the small update may disappear every time.

The parameter can become stuck.

### Solution: Preserve Updates at Higher Precision

One classic mixed-precision training strategy uses:

- A higher-precision master copy of weights.
- Lower-precision representations for suitable forward and backward calculations.
- FP32 for parameter updates or other sensitive operations.

For example:

| Step | FP32 master weight | BF16 compute weight |
|---|---:|---:|
| Initial | 1.0000 | 1.0 |
| After 1 update | 0.9999 | 1.0 |
| After 2 updates | 0.9998 | 1.0 |
| After 100 updates | 0.9900 | 0.98828125 |

The higher-precision master weight preserves accumulated changes.

The BF16 copy changes when enough updates have accumulated to cross a representable-value boundary.

The forward pass still has rounding error, but small parameter updates are not automatically discarded.

**Important qualification:** Not all modern LLM training systems maintain a complete FP32 master copy. Some train with BF16 parameters and use other techniques involving optimizer states, update precision, or rounding.

Mixed precision is a family of strategies, not one fixed implementation.

### Another Problem: Accumulation

Consider:

`1 + 0.001 + 0.001 + ...` (1,000 additions)

The mathematically correct result is:

`2.0`

But if every addition is immediately rounded to BF16:

`1 + 0.001 -> 1`

And the same thing happens repeatedly.

The result may remain 1 rather than growing toward 2.

Using a higher-precision accumulator can preserve these small contributions.

This is why many low-precision matrix multiplication implementations use higher-precision accumulation.

### Why Doesn't Forward-Pass Rounding Destroy Training?

Because the model does not need every intermediate number to be exact.

For many operations, small rounding errors have limited effects on the final loss or predictions.

Optimization can also adapt to the numerical noise introduced by lower precision.

But some operations are more sensitive than others, so mixed-precision systems selectively preserve higher precision.

**Mental model:** Use low precision where the model can tolerate errors, and higher precision where small errors would accumulate, overflow, underflow, or destabilize optimization.

---

## 12. Is Everything We Discussed Called Quantization?

**Question:** Are FP32, FP16, BF16, INT4, and mixed-precision training all examples of quantization?

Not exactly.

### Numerical Representation

This is the broadest topic.

It asks:

**How does a computer represent numbers?**

Examples:

- Integer formats.
- Floating-point formats.
- Sign, exponent, mantissa.
- Rounding and numerical range.

### Quantization

Quantization asks:

**How can we map values from a richer numerical representation into a smaller set of representable levels?**

Example:

`FP32 weight 0.123456 -> INT4 code 12 with scale 0.01 -> 0.12`

Quantization necessarily introduces some approximation for values that cannot be represented exactly.

Converting FP32 values to BF16 can be viewed as quantization in the broad numerical sense.

In LLM engineering, however, *quantization* commonly refers to deliberate low-bit model compression techniques, while BF16/FP16 training is typically discussed separately as reduced- or mixed-precision computation.

### Mixed-Precision Computing

Mixed precision asks:

**Where should we use each numerical precision?**

Example:

- Store some weights in INT4.
- Perform certain operations in BF16.
- Accumulate certain results in FP32.
- Preserve sensitive optimizer state in FP32.

This may involve quantization, but it is not the same concept.

---

## 13. Minimal PyTorch Experiments

### Experiment A: How Much Information Does Each Format Lose?

```python
import torch

x = torch.tensor(1.001)

for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    print(dtype, x.to(dtype).item())
```

Expected observations:

```text
torch.float32   1.0010000467300415
torch.float16   1.0009765625
torch.bfloat16  1.0
```

### Experiment B: Range vs. Precision

```python
import torch

for dtype in [torch.float16, torch.bfloat16, torch.float32]:
    x = torch.tensor(300.0, dtype=dtype)
    tiny = torch.tensor(1e-8, dtype=dtype)

    print(dtype, (x * x).item(), tiny.item())
```

Key observations:

- FP16 overflows for `300 × 300`.
- BF16 retains a finite result, although it rounds it.
- FP16 underflows `1e-8` to zero.
- BF16 and FP32 retain nonzero values close to `1e-8`.

### Experiment C: Why Accumulating Updates Matters

```python
import torch

w32 = torch.tensor(1.0, dtype=torch.float32)
w16 = torch.tensor(1.0, dtype=torch.bfloat16)

for _ in range(100):
    w32 -= 0.0001
    w16 -= 0.0001

print("FP32:", w32.item())
print("BF16:", w16.item())
print("FP32 -> BF16:", w32.to(torch.bfloat16).item())
```

Expected behavior:

- FP32 accumulates the small updates.
- BF16 stays at 1.0 when each tiny update rounds away.
- Converting the accumulated FP32 result to BF16 produces a value below 1.0.

**Lesson:** Rounding once after accumulation is very different from rounding after every operation.

---

## 14. Final Mental Model

| Concept | Fundamental question |
|---|---|
| Bit width | How many distinct codes can we store? |
| Numerical format | What numerical value does each code represent? |
| Exponent | What magnitude can we represent? |
| Mantissa | How finely can we distinguish nearby values? |
| Exponent bias | How can unsigned bits encode negative exponents? |
| Hidden leading 1 | How can we save a predictable bit? |
| Subnormal numbers | How can we represent very small values near zero? |
| Quantization | How can we approximate values with fewer representable levels? |
| Scale | How can we adapt a limited code range to a useful numerical range? |
| NF4 | How can we allocate quantization levels to suit a weight distribution? |
| Mixed precision | Which parts of computation need higher precision? |
| Accumulation precision | How can we preserve information while adding many values? |
| Model sensitivity | Which numerical errors actually affect model behavior? |

### The Five Ideas Worth Remembering

**1. Bits do not inherently mean numbers.**

An encoding rule determines what a bit pattern represents.

**2. Range and precision are different.**

Exponent controls magnitude. Mantissa controls resolution.

**3. Quantization does not try to preserve every parameter exactly.**

It tries to preserve model behavior while reducing representation cost.

**4. Storage precision and computation precision are independent decisions.**

A 4-bit model can perform calculations using wider numerical formats.

**5. Training doesn't require every operation to have the same precision.**

The goal is to use enough precision wherever errors would meaningfully affect learning.

### The Next Fundamental Question

Suppose a matrix has one million weights, and every weight is slightly perturbed by quantization.

Will those errors:

- Accumulate into a huge output error?
- Cancel each other out?
- Become amplified by certain layers?

The answer depends on the matrix, the inputs, the direction of the errors, and the structure of the network.

This leads to the next topic:

**Numerical Error Propagation and Sensitivity in Neural Networks.**

Understanding this is the bridge between knowing how quantization works and understanding *why it works so well for many large neural networks*.